**Author:** Ashutosh Jayant  
**Project:** India State Competitiveness Index (ISCI)

# 03 - Feature Engineering

Harmonises state names to one canonical list, selects the final column and year for
each indicator, derives population-based features, and merges everything into one
master table.

## Setup
Load pandas and add the project folder to the path so `src` can be imported.

In [1]:
import sys
from pathlib import Path
import pandas as pd

root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))

## Harmonise state names
Map every table's state names to the one canonical list, and print any names that did not match.

In [2]:
from src.entities import CANONICAL_ENTITIES, clean_entity

harmonised, flagged = {}, {}
for path in sorted((root / "data" / "processed").glob("*.csv")):
    df = pd.read_csv(path)
    if "state" not in df.columns:
        continue
    df["entity"] = df["state"].map(clean_entity)
    harmonised[path.stem] = df[df["entity"].notna()].copy()
    unmapped = sorted(df.loc[df["entity"].isna(), "state"].unique())
    if unmapped:
        flagged[path.stem] = unmapped

for name, df in harmonised.items():
    print(f"{name:30} entities={df['entity'].nunique()}")

print("\nFlagged (not a canonical entity):")
for name, names in flagged.items():
    print(f"  {name}: {names}")

cd_ratio_sanction              entities=36
cd_ratio_utilisation           entities=36
factories_t116                 entities=36
ger_t1                         entities=36
gfcf_t130                      entities=36
life_expectancy                entities=22
manufacturing_gva              entities=34
msme_udyam                     entities=36
per_capita_nsdp                entities=34
per_capita_power               entities=35
roads                          entities=35
td_losses                      entities=35
telephones                     entities=21
total_gva                      entities=34
total_nsdp                     entities=34
unemployment_rural_overall     entities=35
unemployment_urban_overall     entities=35

Flagged (not a canonical entity):
  cd_ratio_sanction: ['*: Since']
  cd_ratio_utilisation: ['*: Since']
  td_losses: ['Jammu & Kashmir and Ladakh']
  telephones: ['-: Not Available. #: Data till Nov 30th,', 'Chennai**', 'Kolkata**', 'Mumbai', 'North East-I', 'North E

## Data quality report
Count how many states each indicator covers and which year, then save the result.

In [3]:
N = len(CANONICAL_ENTITIES)

def coverage(entities, year, note=""):
    n = len(set(entities))
    return {"entities": n, "missing": N - n, "year": str(year), "notes": note}

report = {}

yearly = ["factories_t116", "gfcf_t130", "per_capita_power", "roads", "td_losses",
          "manufacturing_gva", "total_gva", "per_capita_nsdp", "total_nsdp",
          "cd_ratio_utilisation", "life_expectancy",
          "unemployment_rural_overall", "unemployment_urban_overall"]
for name in yearly:
    df = harmonised[name]
    year = df["year"].max()
    present = df.loc[df["year"] == year, "entity"].unique()
    report[name] = coverage(present, year)

g = harmonised["ger_t1"]
g = g[(g["level"] == "Secondary") & (g["gender"] == "Total")]
year = g["year"].max()
report["ger_secondary_total"] = coverage(g.loc[g["year"] == year, "entity"].unique(), year)

report["msme_udyam"] = coverage(harmonised["msme_udyam"]["entity"].unique(), "2026")
report["telephones"] = {"entities": harmonised["telephones"]["entity"].nunique(),
                        "missing": "-", "year": "-", "notes": "dropped in v1.0 (telecom circles)"}
report["life_expectancy"]["notes"] = "major states only"

dqr = pd.DataFrame(report).T[["entities", "missing", "year", "notes"]]
dqr.to_csv(root / "data" / "processed" / "data_quality_report.csv")
print(dqr.to_string())

                           entities missing     year                              notes
factories_t116                   36       0  2023-24                                   
gfcf_t130                        36       0  2023-24                                   
per_capita_power                 34       2  2024-25                                   
roads                            35       1     2020                                   
td_losses                        34       2  2022-23                                   
manufacturing_gva                34       2  2024-25                                   
total_gva                        34       2  2024-25                                   
per_capita_nsdp                  34       2  2024-25                                   
total_nsdp                       34       2  2024-25                                   
cd_ratio_utilisation             36       0     2025                                   
life_expectancy                 

## Pick one value per state
For each indicator choose the latest year with enough coverage, take one value per state, and record the year used.

In [4]:
def select_year(df, value_col="value", threshold=0.90):
    cov = df[df[value_col].notna()].groupby("year")["entity"].nunique()
    ok = cov[cov >= threshold * cov.max()]
    return max(ok.index)

def reduce_indicator(df, agg, value_col="value", year=None):
    if year is None:
        year = select_year(df, value_col)
    rows = df[df["year"] == year]
    if agg == "sum":
        return year, rows.groupby("entity")[value_col].sum(min_count=1)
    return year, rows.groupby("entity")[value_col].mean()

# NSDP and GVA family: pinned to 2022-23, the latest year with full state coverage.
# 2023-24 omits Gujarat and 2024-25 is provisional with lower coverage.
PINNED = {"total_nsdp", "per_capita_nsdp", "total_gva", "manufacturing_gva", "gfcf_t130"}
PINNED_YEAR = "2022-23"

sums = {"factories_t116": "factories", "gfcf_t130": "gfcf", "roads": "roads_km",
        "manufacturing_gva": "manufacturing_gva", "total_gva": "total_gva",
        "total_nsdp": "total_nsdp"}
means = {"per_capita_power": "per_capita_power", "td_losses": "td_losses",
         "cd_ratio_utilisation": "cd_ratio", "per_capita_nsdp": "per_capita_nsdp",
         "life_expectancy": "life_expectancy",
         "unemployment_rural_overall": "unemp_rural",
         "unemployment_urban_overall": "unemp_urban"}

values, meta = {}, []
for src, col in sums.items():
    year, s = reduce_indicator(harmonised[src], "sum",
                               year=PINNED_YEAR if src in PINNED else None)
    values[col] = s
    meta.append((col, year))
for src, col in means.items():
    year, s = reduce_indicator(harmonised[src], "mean",
                               year=PINNED_YEAR if src in PINNED else None)
    values[col] = s
    meta.append((col, year))

g = harmonised["ger_t1"]
g = g[(g["level"] == "Secondary") & (g["gender"] == "Total")]
year, s = reduce_indicator(g, "mean", "ger")
values["ger"] = s
meta.append(("ger", year))

values["msme_registrations"] = harmonised["msme_udyam"].groupby("entity")["total"].sum(min_count=1)
meta.append(("msme_registrations", "2026"))

selected = pd.DataFrame(values).reindex(CANONICAL_ENTITIES)

metadata = pd.DataFrame(meta, columns=["indicator", "year_used"])
metadata["entities"] = [selected[c].notna().sum() for c in metadata["indicator"]]
metadata["coverage_pct"] = (metadata["entities"] / len(CANONICAL_ENTITIES) * 100).round().astype(int)
metadata.to_csv(root / "data" / "processed" / "indicator_metadata.csv", index=False)
print(metadata.to_string(index=False))

         indicator year_used  entities  coverage_pct
         factories   2023-24        36           100
              gfcf   2022-23        35            97
          roads_km      2020        35            97
 manufacturing_gva   2022-23        34            94
         total_gva   2022-23        34            94
        total_nsdp   2022-23        34            94
  per_capita_power   2024-25        34            94
         td_losses   2022-23        33            92
          cd_ratio      2025        36           100
   per_capita_nsdp   2022-23        34            94
   life_expectancy   2019-23        22            61
       unemp_rural   2023-24        34            94
       unemp_urban   2023-24        35            97
               ger   2024-25        36           100
msme_registrations      2026        36           100


## Derived features and master table
Calculate population, the density features and the share features, then save the master table.

In [5]:
selected["population"] = selected["total_nsdp"] / selected["per_capita_nsdp"]
selected["factory_density"] = selected["factories"] / selected["population"]
selected["msme_density"] = selected["msme_registrations"] / selected["population"]
selected["road_density"] = selected["roads_km"] / selected["population"]
selected["manufacturing_share"] = selected["manufacturing_gva"] / selected["total_gva"] * 100
selected["investment_rate"] = selected["gfcf"] / selected["total_gva"] * 100

index_features = ["ger", "life_expectancy", "unemp_rural", "unemp_urban", "cd_ratio",
                  "per_capita_power", "td_losses", "road_density", "factory_density",
                  "msme_density", "manufacturing_share", "investment_rate"]

master = selected[index_features + ["population", "per_capita_nsdp"]]
master.to_csv(root / "data" / "processed" / "master_features.csv")
print(master.notna().sum().to_string())
print()
print(master.loc[["Bihar", "Maharashtra", "Kerala", "Delhi"]].round(2).to_string())

ger                    36
life_expectancy        22
unemp_rural            34
unemp_urban            35
cd_ratio               36
per_capita_power       34
td_losses              33
road_density           33
factory_density        34
msme_density           34
manufacturing_share    34
investment_rate        34
population             34
per_capita_nsdp        34

              ger  life_expectancy  unemp_rural  unemp_urban  cd_ratio  per_capita_power  td_losses  road_density  factory_density  msme_density  manufacturing_share  investment_rate  population  per_capita_nsdp
entity                                                                                                                                                                                                             
Bihar        44.7             69.1         26.0         73.0      53.5             343.9       25.8        237.78             2.69       3285.36                 7.97             1.64     1259.90          54637.0